In [18]:
import argparse
import json
import tensorboard
import tensorboardX
import os
import argparse
import json
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim 
import nni
from nni.nas.nn.pytorch import ModelSpace, LayerChoice, MutableConv2d, MutableBatchNorm2d, MutableReLU,MutableLinear
from nni.nas.experiment.config import NasExperimentConfig
from pytorch_lightning import Trainer
from nni.nas.evaluator.pytorch import Lightning, ClassificationModule, Trainer
from nni.nas.experiment import NasExperiment
from nni.nas.space import model_context
from nni.nas.hub.pytorch import DARTS
from nni.nas.strategy import DARTS as DartsStrategy
from pytorch_lightning.loggers import TensorBoardLogger
from torch.utils.data import DataLoader
from torch.utils.data.sampler import SubsetRandomSampler
from torchvision import transforms
from torchvision.datasets import CIFAR10
from nni.nas.experiment import NasExperiment
from nni.nas.evaluator import FunctionalEvaluator
from nni.nas.evaluator import FunctionalEvaluator
import nni.nas.strategy as strategy
from torchvision import transforms
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
from nni.experiment.config import utils, ExperimentConfig
#from ops import AvgPool,DilConv,SepConv
import genotypes
from pytorch_lightning.callbacks import ModelCheckpoint
torch.set_float32_matmul_precision('medium')
from tqdm import tqdm
from nni.nas.nn.pytorch import LayerChoice, ModelSpace,ValueChoice
from torch.utils.data import DataLoader, Dataset, SubsetRandomSampler
from pytorch_lightning import LightningModule, Trainer
from torchvision import datasets, transforms
from nni.nas.evaluator.pytorch import Classification
import torch
import torchvision.transforms.functional as F
import torchvision.transforms as transforms
import numpy as np
import cv2
from PIL import Image
from torch.utils.data import DataLoader, random_split
from torchvision import datasets
import matplotlib.pyplot as plt
from nni.nas.evaluator.pytorch import Lightning, Trainer
import torchvision


import numpy as np
import cv2
from PIL import Image

In [19]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from tensorflow import keras
from PIL import Image
import os
import pathlib
import random as rn
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, array_to_img, load_img
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.applications import VGG19,VGG16, ResNet50
from tensorflow.keras.layers import Conv2D, MaxPool2D, Dense, Flatten, Dropout
from sklearn.metrics import accuracy_score
import argparse


In [20]:
data_dir = 'C:/Users/senti/Desktop/GTSDB'
train_path = 'C:/Users/senti/Desktop/GTSDB/Train'
test_path = 'C:/Users/senti/Desktop/GTSDB/Test'
height = 32
width = 32

In [21]:
classes = { 0:'Speed limit (20km/h)',
            1:'Speed limit (30km/h)', 
            2:'Speed limit (50km/h)', 
            3:'Speed limit (60km/h)', 
            4:'Speed limit (70km/h)', 
            5:'Speed limit (80km/h)', 
            6:'End of speed limit (80km/h)', 
            7:'Speed limit (100km/h)', 
            8:'Speed limit (120km/h)', 
            9:'No passing', 
            10:'No passing veh over 3.5 tons', 
            11:'Right-of-way at intersection', 
            12:'Priority road', 
            13:'Yield', 
            14:'Stop', 
            15:'No vehicles', 
            16:'Veh > 3.5 tons prohibited', 
            17:'No entry', 
            18:'General caution', 
            19:'Dangerous curve left', 
            20:'Dangerous curve right', 
            21:'Double curve', 
            22:'Bumpy road', 
            23:'Slippery road', 
            24:'Road narrows on the right', 
            25:'Road work', 
            26:'Traffic signals', 
            27:'Pedestrians', 
            28:'Children crossing', 
            29:'Bicycles crossing', 
            30:'Beware of ice/snow',
            31:'Wild animals crossing', 
            32:'End speed + passing limits', 
            33:'Turn right ahead', 
            34:'Turn left ahead', 
            35:'Ahead only', 
            36:'Go straight or right', 
            37:'Go straight or left', 
            38:'Keep right', 
            39:'Keep left', 
            40:'Roundabout mandatory', 
            41:'End of no passing', 
            42:'End no passing veh > 3.5 tons' }

In [22]:
batch_size = 32

In [23]:
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

class GTSDBTrainDataset(Dataset):
    def __init__(self, img_dir, gt_file, transform=None):
        self.img_dir = img_dir
        self.transform = transform

        # Load gt.txt
        columns = ['filename', 'x1', 'y1', 'x2', 'y2', 'label']
        self.annotations = pd.read_csv(gt_file, sep=';', names=columns)

        # Keep only images that exist
        self.annotations = self.annotations[self.annotations['filename'].apply(lambda x: os.path.exists(os.path.join(img_dir, x)))]

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        # Get row data
        row = self.annotations.iloc[idx]
        img_path = os.path.join(self.img_dir, row['filename'])
        label = int(row['label'])
        x1, y1, x2, y2 = row[['x1', 'y1', 'x2', 'y2']].astype(int)

        # Load image
        image = Image.open(img_path).convert("RGB")

        # Crop image based on bounding box
        cropped_image = image.crop((x1, y1, x2, y2))

        # Apply transformations
        if self.transform:
            cropped_image = self.transform(cropped_image)

        return cropped_image, label

# Define transformations
transform = transforms.Compose([
    transforms.RandomRotation(15),   # Rotate up to ±15 degrees
    transforms.RandomHorizontalFlip(),  # Flip image with 50% probability
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),  # Random color change
    transforms.Resize((32, 32)),  # Resize to 32x32
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])


# Dataset and DataLoader
train_dataset = GTSDBTrainDataset(
    img_dir="C:/Users/senti/Desktop/GTSDB/Train",
    gt_file="C:/Users/senti/Desktop/GTSDB/gt.txt",
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)


# Check dataset size
print(f"Train dataset size: {len(train_dataset)}")


Train dataset size: 852


In [24]:
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

class GTSDBTestDataset(Dataset):
    def __init__(self, img_dir, gt_file, transform=None):
        self.img_dir = img_dir
        self.transform = transform

        # Load gt.txt
        columns = ['filename', 'x1', 'y1', 'x2', 'y2', 'label']
        self.annotations = pd.read_csv(gt_file, sep=';', names=columns)

        # Keep only images that exist
        self.annotations = self.annotations[self.annotations['filename'].apply(lambda x: os.path.exists(os.path.join(img_dir, x)))]

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        # Get image filename and label
        row = self.annotations.iloc[idx]
        img_path = os.path.join(self.img_dir, row['filename'])
        label = int(row['label'])

        # Load image
        image = Image.open(img_path).convert("RGB")

        # Apply transformations
        if self.transform:
            image = self.transform(image)

        return image, label

# Define transformations
transform = transforms.Compose([
    transforms.Resize((32, 32)),  # Resize to 32x32
    transforms.ToTensor(),        # Convert to Tensor
    transforms.Normalize((0.5,), (0.5,))  # Normalize
])

# Dataset and DataLoader for Test
test_dataset = GTSDBTestDataset(
    img_dir="C:/Users/senti/Desktop/GTSDB/Test",
    gt_file="C:/Users/senti/Desktop/GTSDB/gt.txt",
    transform=transform
)

test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

# Check dataset size
print(f"Test dataset size: {len(test_dataset)}")


Test dataset size: 476


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)  # Move batch to GPU
    # Now you can use images and labels for training


In [9]:
@nni.trace
class DartsClassificationModule(ClassificationModule):
    def __init__( self,learning_rate: float = 0.001,weight_decay: float = 0.,auxiliary_loss_weight: float = 0.4,max_epochs: int = 600):
        super().__init__(learning_rate=learning_rate, weight_decay=weight_decay, export_onnx=False, num_classes=43)
        
        self.auxiliary_loss_weight = auxiliary_loss_weight
        self.max_epochs = max_epochs
        self.learning_rate = learning_rate


    def configure_optimizers(self):
        optimizer = torch.optim.SGD(self.parameters(), lr=self.learning_rate, momentum=0.9, weight_decay=0.)
        return {
            'optimizer': optimizer,
            'lr_scheduler': torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)
        }

    def training_step(self, batch, batch_idx):
        x, y = batch
        y = y.long()  # Ensure labels are integer (not float or one-hot)
        
        if self.auxiliary_loss_weight:
            y_hat, y_aux = self(x)
            loss_main = self.criterion(y_hat, y)
            loss_aux = self.criterion(y_aux, y)
            self.log('train_loss_main', loss_main)
            self.log('train_loss_aux', loss_aux)
            loss = loss_main + self.auxiliary_loss_weight * loss_aux
        else:
            y_hat = self(x)
            loss = self.criterion(y_hat, y)
        
        self.log('train_loss', loss, prog_bar=True)
        
        for name, metric in self.metrics.items():
            self.log('train_' + name, metric(y_hat, y), prog_bar=True)
        
        return loss


    def on_train_epoch_start(self):
        # Set drop path probability before every epoch. This has no effect if drop path is not enabled in model.
        #self.model.set_drop_path_prob(self.model.drop_path_prob * self.current_epoch / self.max_epochs)

        # Logging learning rate at the beginning of every epoch
        self.log('lr', self.trainer.optimizers[0].param_groups[0]['lr'])


In [10]:

class CustomDARTSSpace(ModelSpace):
    def __init__(self, input_channels=3, channels=64, num_classes=43, layers=7,verbose =0, drop_path_prob = 0.1):
        super(CustomDARTSSpace, self).__init__()

        #________________________________________________________________________________________________________________________
        #Inizialization
        self.layers = nn.ModuleList()
        self.drop_path_prob = drop_path_prob
        self.verbose = verbose


        #________________________________________________________________________________________________________________________
        #Channel choices
        layer0_out = 16
        layer1_out = nni.choice('layer1_out_channels', [16,32,64])
        layer2_out= nni.choice('layer2_out_channels', [16,32,64])
        layer3_out= nni.choice('layer3_out_channels', [16,32,64])
        layer4_out = nni.choice('layer4_out_channels', [16,32,64])
        layer5_out= nni.choice('layer5_out_channels', [16,32,64])
        layer6_out= nni.choice('layer6_out_channels', [16,32,64])
        layer7_out= 22
        
        #________________________________________________________________________________________________________________________
        #Layer 0
        self.preliminary_layer = nn.Conv2d(3, layer0_out, kernel_size=3, padding=0, bias=False)
        self.layer0_bn = torch.nn.BatchNorm2d(layer0_out)
        self.layer0_relu = torch.nn.ReLU(inplace=True)
        
        #________________________________________________________________________________________________________________________
        #Layer 1
        layer1 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1 ),
                MutableConv2d(layer0_out, layer1_out, kernel_size=3),
                MutableBatchNorm2d(layer1_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer0_out, layer1_out, kernel_size=3),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer1_out),
                MutableReLU()
            )
        ], label='layer_1')
        self.layers.append(layer1)
        
        #________________________________________________________________________________________________________________________
        #Layer 2
        layer2 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer1_out, layer2_out, kernel_size=3),
                MutableBatchNorm2d(layer2_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer1_out, layer2_out, kernel_size=3),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer2_out),
                MutableReLU()
            )
        ], label='layer_2')
        self.layers.append(layer2)
        
        #________________________________________________________________________________________________________________________
        #Layer 3
        layer3 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer2_out, layer3_out, kernel_size=3),
                MutableBatchNorm2d(layer3_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer2_out, layer3_out, kernel_size=3),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer3_out),
                MutableReLU()
            )
        ], label='layer_3')
        self.layers.append(layer3)
                #________________________________________________________________________________________________________________________
        #Layer 4
        layer4 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer3_out, layer4_out, kernel_size=3),
                MutableBatchNorm2d(layer4_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer3_out, layer4_out, kernel_size=3),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer4_out),
                MutableReLU()
            )
        ], label='layer_4')
        self.layers.append(layer4)
                #________________________________________________________________________________________________________________________
        #Layer 5
        layer5 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer4_out, layer5_out, kernel_size=3),
                MutableBatchNorm2d(layer5_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer4_out, layer5_out, kernel_size=3),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer5_out),
                MutableReLU()
            )
        ], label='layer_5')
        self.layers.append(layer5)
                #________________________________________________________________________________________________________________________
        #Layer 6
        layer6= LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer5_out, layer6_out, kernel_size=3),
                MutableBatchNorm2d(layer6_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer5_out, layer6_out, kernel_size=3),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer6_out),
                MutableReLU()
            )
        ], label='layer_6')
        self.layers.append(layer6)
                #________________________________________________________________________________________________________________________
        #Layer 7
        layer7 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer6_out, layer7_out, kernel_size=3),
                MutableBatchNorm2d(layer7_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer6_out, layer7_out, kernel_size=3),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer7_out),
                MutableReLU()
            )
        ], label='layer_7')
        self.layers.append(layer7)
        

        
        #________________________________________________________________________________________________________________________
        #Linear
        self.pool = nn.AdaptiveAvgPool2d((3, 3))
        feature1 = nni.choice('feature1', [32, 64, 128])
        feature2 = nni.choice('feature2', [32 ,64, 128])
        feature3 = nni.choice('feature3', [32, 64])
        self.fc1 = MutableLinear(198, feature1) 
        self.fc2 = MutableLinear(feature1, feature2) 
        self.fc3 = MutableLinear(feature2, feature3)  
        self.relu = nn.ReLU()
        self.classifier = MutableLinear(feature3, 43)

    def forward(self, x):
        #________________________________________________________________________________________________________________________
        #Layer 0
        x = self.preliminary_layer(x)
        X = self.layer0_bn(x)
        x = self.layer0_relu(x)
        if self.verbose == 1 :
            print(f'After preliminary layer: {x.shape}')
        #________________________________________________________________________________________________________________________
        #Layer 1 to n
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if self.verbose == 1 :
                print(f'After layer {i+1}: {x.shape}')
            if i == 1 or i == 3 or i == 6:
                x = nn.AvgPool2d(kernel_size=2, stride=2)(x)
                if self.verbose == 1 :
                    print(f'After avg pooling: {x.shape}')
        
        #________________________________________________________________________________________________________________________
        #Adaprive pool
        x =  self.pool(x)
        if self.verbose == 1 :
            print(f'After adaptive pooling: {x.shape}')

        #________________________________________________________________________________________________________________________
        #Flatten
        x = torch.flatten(x, 1)
        if self.verbose == 1 :
            print(f'After flattening: {x.shape}')
        #________________________________________________________________________________________________________________________
        
        x = self.fc1(x)
        x= self.relu(x)
        if self.verbose == 1 :
            print(f'After fc1: {x.shape}')
        x = self.fc2(x)
        x= self.relu(x)
        if self.verbose == 1 :
            print(f'After fc2: {x.shape}')
        x = self.fc3(x)
        x= self.relu(x)
        if self.verbose == 1 :
            print(f'After fc3: {x.shape}')
        #________________________________________________________________________________________________________________________
        #Classification 
        x = self.classifier(x)

        
        if self.verbose == 1 :
            print(f'After classifier: {x.shape}')
        #self.first_iter = False
        return x

    def set_drop_path_prob(self, drop_path_prob):
        self.drop_path_prob = drop_path_prob
        for layer in self.layers:
            if hasattr(layer, 'set_drop_path_prob'):
                layer.set_drop_path_prob(drop_path_prob)


In [12]:
# Checkpoint 
checkpoint_callback = ModelCheckpoint(
    monitor='train_acc', 
    dirpath='./checkpoints',
    filename='GTSDB-best-checkpoint',
    save_top_k=1,
    mode='max'
    
)
max_epochs = 600

evaluator = Lightning(
    DartsClassificationModule(1e-2, 0., 0., max_epochs),
    Trainer(
        accelerator="auto",
        callbacks=[checkpoint_callback],
        max_epochs=max_epochs
    ),
    train_dataloaders=train_loader,
    val_dataloaders=test_loader
)
strategy = DartsStrategy(gradient_clip_val=0.)
def search(log_dir: str, batch_size: int = 64):

    # Define model search space
    model_space = CustomDARTSSpace(input_channels=3, channels=64, num_classes=43, layers=7, verbose=0)
    model_space.set_drop_path_prob(0.)

    # Run NAS experiment
    exp_config = NasExperimentConfig.default(model_space, evaluator, strategy)
    exp_config.experiment_working_directory = "./DartsCheckpoints"
    exp_config.experiment_name = "Darts_search"
    exp_config.trial_concurrency = 1
    experiment = NasExperiment(model_space, evaluator, strategy, config = exp_config)
    experiment.run()

    return experiment


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [13]:
experiment_results = search("./",32)

[2025-03-05 18:34:27] Config is not provided. Will try to infer.
[2025-03-05 18:34:27] Strategy is found to be a one-shot strategy. Setting execution engine to "sequential" and format to "raw".
[2025-03-05 18:34:27] WARNING: `training_service` will be ignored for sequential execution engine.
[2025-03-05 18:34:27] WARNING: `training_service` will be ignored for sequential execution engine.
[2025-03-05 18:34:27] WARNING: `training_service` will be ignored for sequential execution engine.
[2025-03-05 18:34:27] WARNING: `training_service` will be ignored for sequential execution engine.
[2025-03-05 18:34:27] WARNING: `training_service` will be ignored for sequential execution engine.
[2025-03-05 18:34:27] WARNING: `training_service` will be ignored for sequential execution engine.
[2025-03-05 18:34:27] WARNING: `training_service` will be ignored for sequential execution engine.
[2025-03-05 18:34:27] WARNING: `training_service` will be ignored for sequential execution engine.
[2025-03-05 18

C:\Users\senti\anaconda3\envs\NNI_NAS_local\Lib\site-packages\pytorch_lightning\callbacks\model_checkpoint.py:653: Checkpoint directory C:\Users\senti\Documents\GitHub\PhotonicNas\checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name            | Type                      | Params
--------------------------------------------------------------
0 | training_module | DartsClassificationModule | 468 K 
--------------------------------------------------------------
468 K     Trainable params
0         Non-trainable params
468 K     Total params
1.874     Total estimated model params size (MB)
C:\Users\senti\anaconda3\envs\NNI_NAS_local\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:441: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=21` in the `DataLoader` to improve performance.
C:\Users\senti\anaconda3\envs\NNI_NAS_local\Li

Epoch 0:   4%|▎         | 1/27 [00:00<00:20,  1.25it/s, v_num=168, train_loss=3.800, train_acc=0.000]

C:\Users\senti\anaconda3\envs\NNI_NAS_local\Lib\site-packages\torch\autograd\graph.py:744: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at ..\aten\src\ATen\native\cudnn\Conv_v8.cpp:919.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 0:  56%|█████▌    | 15/27 [00:07<00:06,  1.98it/s, v_num=168, train_loss=3.760, train_acc=0.000] 

...

Epoch 1:   0%|          | 0/27 [00:00<?, ?it/s, v_num=168, train_loss=3.730, train_acc=0.100]          

C:\Users\senti\anaconda3\envs\NNI_NAS_local\Lib\site-packages\torch\autograd\graph.py:744: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at ..\aten\src\ATen\native\cudnn\Conv_v8.cpp:919.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 599: 100%|██████████| 27/27 [00:13<00:00,  2.07it/s, v_num=168, train_loss=1.320, train_acc=0.450]

`Trainer.fit` stopped: `max_epochs=600` reached.


Epoch 599: 100%|██████████| 27/27 [00:13<00:00,  2.07it/s, v_num=168, train_loss=1.320, train_acc=0.450]
[2025-03-05 20:46:23] Waiting for models submitted to engine to finish...
[2025-03-05 20:46:23] Experiment is completed.
[2025-03-05 20:46:23] WARNING: `training_service` will be ignored for sequential execution engine.


In [16]:
checkpoint_path = './checkpoints/GTSDB-best-checkpoint-v1.ckpt'

checkpoint = torch.load(checkpoint_path, map_location=torch.device('cpu'))

checkpoint_model = CustomDARTSSpace(input_channels=3, channels=64, num_classes=43, layers=7,verbose =0)

checkpoint_state_dict = checkpoint_model.state_dict()
pretrained_state_dict = checkpoint['state_dict']

if 'global_step' in checkpoint:
    print("Global Step:", checkpoint['global_step'])

if 'callbacks' in checkpoint and isinstance(checkpoint['callbacks'], dict):
    for key, callback in checkpoint['callbacks'].items():
        if isinstance(callback, dict) and 'best_model_score' in callback:
            print("Best Model Score (Train Accuracy):", callback['best_model_score'].item())

Global Step: 6912
Best Model Score (Train Accuracy): 1.0


In [14]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
import torch
torch.cuda.empty_cache()
